In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset,DataLoader
from keras.datasets import reuters
from keras.utils import pad_sequences

In [2]:
num_words=10_000
max_len=120

In [3]:
from pathlib import Path

In [4]:
dataset_path=Path.cwd()/"reuters_dataset"

In [5]:
import os
os.environ["KERAS_HOME"] = "/content/reuters_dataset"

In [6]:
(x_train,y_train),(x_test,y_test)=reuters.load_data(num_words=10_000)

In [7]:
print(x_train.shape, x_test.shape)

(8982,) (2246,)


In [8]:
x_train=pad_sequences(x_train,maxlen=max_len)
x_test=pad_sequences(x_test,maxlen=max_len)

In [9]:
x_train=torch.tensor(x_train,dtype=torch.long)
y_train=torch.tensor(y_train,dtype=torch.long)
x_test=torch.tensor(x_test,dtype=torch.long)
y_test=torch.tensor(y_test,dtype=torch.long)

In [10]:
train_dataset=TensorDataset(x_train,y_train)
test_dataset=TensorDataset(x_test,y_test)
train_loader=DataLoader(train_dataset,shuffle=True,batch_size=64)
test_loader=DataLoader(test_dataset,shuffle=False,batch_size=32)


In [11]:
from torch.storage import HAS_NUMPY
class BiLSTM(nn.Module):
  def __init__(self,vocab_size=10000,embed_dim=128,hidden_dim=64,num_classes=46):
    super().__init__()
    self.embedding=nn.Embedding(vocab_size,embed_dim,padding_idx=0)
    self.lstm=nn.LSTM(embed_dim,hidden_dim,batch_first=True,bidirectional=True)
    self.fc = nn.Linear(hidden_dim * 2, num_classes)
  def forward(self,x):
    embed=self.embedding(x)
    _,(h_n,_)=self.lstm(embed)
    h=torch.cat((h_n[-2],h_n[-1]),dim=1)
    return self.fc(h)

In [12]:
model=BiLSTM(vocab_size=num_words,num_classes=46)


In [13]:
criterion=nn.CrossEntropyLoss()
optimizer=torch.optim.Adam(model.parameters(),lr=0.001)

In [14]:
epochs=2

In [15]:
for epoch in range(epochs):
  model.train()
  total_loss,correct,total=0,0,0
  for x_b,y_b in train_loader:
    optimizer.zero_grad()
    output=model(x_b)
    loss=criterion(output,y_b)
    loss.backward()
    optimizer.step()
    total_loss+=loss.item()
    correct+=(output.argmax(dim=1)==y_b).sum().item()
    total+=y_b.size(0)
  print(f"Epoch: {epoch+1/epochs} | Loss: {total_loss} | Correct: {correct} | Total: {total}")



Epoch: 0.5 | Loss: 340.5898388624191 | Correct: 3533 | Total: 8982
Epoch: 1.5 | Loss: 273.0785571336746 | Correct: 4508 | Total: 8982


In [16]:
model.eval()

BiLSTM(
  (embedding): Embedding(10000, 128, padding_idx=0)
  (lstm): LSTM(128, 64, batch_first=True, bidirectional=True)
  (fc): Linear(in_features=128, out_features=46, bias=True)
)

In [17]:
with torch.no_grad():
  pred=model(x_test[0].unsqueeze(0))

correct=(pred.argmax(dim=1)==y_test).sum().item()
total=y_test.size(0)
model.eval()
correct, total = 0, 0
with torch.no_grad():
    for xb, yb in test_loader:
        preds = model(xb).argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

print(f"Correct: {correct} | Incorrect: {total - correct} | Accuracy: {100.0*correct/total:.2f}%")

Correct: 1099 | Incorrect: 1147 | Accuracy: 48.93%


In [18]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader, random_split
from torch.utils.tensorboard import SummaryWriter

num_classes = 46

# Train / validation split (90% / 10%)
full_train = TensorDataset(x_train, y_train)
n_val = int(0.1 * len(full_train))
train_part, val_part = random_split(
    full_train, [len(full_train) - n_val, n_val],
    generator=torch.Generator().manual_seed(42)
)
tb_train_loader = DataLoader(train_part, batch_size=64, shuffle=True)
tb_val_loader   = DataLoader(val_part,   batch_size=64, shuffle=False)

# Fresh model
model = BiLSTM(vocab_size=num_words, num_classes=num_classes)
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-3)

# 4. Store logs under ./runs/
writer = SummaryWriter("./runs/")

# 1 & 2. Train for 2 epochs, log training loss and validation accuracy
for epoch in range(2):
    model.train()
    total_loss, total = 0.0, 0
    for xb, yb in tb_train_loader:
        optimizer.zero_grad()
        out = model(xb)
        loss = criterion(out, yb)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * yb.size(0)
        total += yb.size(0)
    train_loss = total_loss / total

    model.eval()
    correct, vtotal = 0, 0
    with torch.no_grad():
        for xb, yb in tb_val_loader:
            correct += (model(xb).argmax(dim=1) == yb).sum().item()
            vtotal += yb.size(0)
    val_acc = 100.0 * correct / vtotal

    writer.add_scalar("Loss/train", train_loss, epoch)
    writer.add_scalar("Accuracy/val", val_acc, epoch)
    print(f"Epoch {epoch+1}/2 | Train Loss: {train_loss:.4f} | Val Acc: {val_acc:.2f}%")

# 3. Add one batch of text embeddings
xb, yb = next(iter(tb_train_loader))
model.eval()
with torch.no_grad():
    feats = model.embedding(xb).mean(dim=1)     # [64, 128] sentence embeddings

writer.add_embedding(
    feats,
    metadata=[str(v) for v in yb.tolist()],
    tag="news_embeddings"
)

writer.close()
print("Logs saved to ./runs/")

Epoch 1/2 | Train Loss: 2.4354 | Val Acc: 46.10%
Epoch 2/2 | Train Loss: 1.9447 | Val Acc: 50.11%
Logs saved to ./runs/
